# A/A Test: Flash Sales City Lookalike (Hourly BSTS)

Validates the City Lookalike synthetic control under **Flash Sales conditions**:
- Hourly granularity (dinner hours 16:00-22:00)
- 2-day post-period (12 hourly observations)
- 6-week pre-period (252 hourly observations)

Expected result: **zero uplift**.

| Verdict | Condition |
|---------|----------|
| PASS | |mean uplift| <= 2% AND 95% CI contains zero |
| WARNING | |mean uplift| > 2% OR 95% CI excludes zero |
| HARD FAIL | |mean uplift| > 5% |

## 1. Config & Authentication

In [ ]:
# Step 1: Install tfcausalimpact + fix numpy/pandas compatibility
# Run this cell FIRST. Runtime will restart automatically.
# After restart, SKIP this cell and run the next one.
!pip install -q tfcausalimpact
!pip install -q pandas numpy --upgrade --force-reinstall
import os
os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import time, gc, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)

import aa_config as cfg
import flash_sales_utils as fsu

client = bigquery.Client(project=cfg.PROJECT_ID)
hour_start, hour_end = cfg.FLASH_SALES_HOUR_WINDOWS[cfg.FLASH_SALES_DEFAULT_WINDOW]
print(f'Connected to {cfg.PROJECT_ID}')
print(f'Treatment cities: {cfg.FLASH_SALES_TREATMENT_CITIES}')
print(f'Hour window: {hour_start}:00 - {hour_end}:00')
print(f'A/A windows: {len(cfg.FLASH_SALES_AA_WINDOWS)}')
print(f'Total runs: {len(cfg.FLASH_SALES_AA_WINDOWS) * len(cfg.FLASH_SALES_TREATMENT_CITIES.get("DE", []))}')

## 2. Run A/A Test

In [ ]:
start_time = time.time()
results = []
run = 0
cities_de = cfg.FLASH_SALES_TREATMENT_CITIES.get('DE', [])
total = len(cfg.FLASH_SALES_AA_WINDOWS) * len(cities_de)

for window in cfg.FLASH_SALES_AA_WINDOWS:
    print(f'\n=== City features for DE (ref: {window["post_start"]}) ===')
    df_cities = fsu.get_city_features(client, 'DE', window['post_start'])
    print(f'  {len(df_cities)} cities')

    for treatment_city in cities_de:
        run += 1
        print(f'\n=== [{run}/{total}] {treatment_city} | '
              f'{window["post_start"]}-{window["post_end"]} ===')

        # Step 1: Find control cities
        controls = fsu.find_control_cities(
            df_cities, treatment_city, cfg.FLASH_SALES_KNN_NEIGHBORS)
        if not controls:
            print('  No controls found, skipping.')
            continue
        print(f'  KNN: {len(controls)} candidates')

        # Step 2: Query hourly orders
        all_cities = [treatment_city] + controls
        df_orders = fsu.get_hourly_orders(
            client, 'DE', all_cities,
            window['pre_start'], window['post_end'],
            hour_start, hour_end)
        print(f'  Hourly orders: {len(df_orders)} rows')

        if df_orders.empty:
            print('  No order data, skipping.')
            continue

        # Step 3: Correlation filter (aggregate hourly to weekly)
        df_pre = df_orders[
            pd.to_datetime(df_orders['order_hour']) <
            pd.to_datetime(window['post_start'])
        ]
        filtered = fsu.apply_correlation_filter(
            df_pre, treatment_city, controls,
            cfg.FLASH_SALES_CORRELATION_THRESHOLD)
        if not filtered:
            print('  No controls pass correlation filter, skipping.')
            continue

        # Step 4: Contamination check
        contam = fsu.check_control_contamination(
            client, 'DE', filtered,
            window['post_start'], window['post_end'])
        n_contaminated = contam['city'].nunique() if len(contam) > 0 else 0

        # Step 5: Build hourly pivot and run CausalImpact
        try:
            pivot, pre_period, post_period = fsu.build_hourly_pivot(
                df_orders, treatment_city, filtered,
                window['pre_start'], window['pre_end'],
                window['post_start'], window['post_end'])

            result = fsu.run_causal_impact(pivot, pre_period, post_period)

            if result:
                incr = result['actual_orders'] - result['predicted_orders']
                row = {
                    'technique': cfg.FLASH_SALES_TECHNIQUE,
                    'country': 'DE',
                    'window_start': window['post_start'],
                    'window_end': window['post_end'],
                    'seed': None,
                    'treatment_city': treatment_city,
                    'base_segment': 'total',
                    'n_units': 1,
                    'n_controls': len(filtered),
                    'exact_match_pct': None,
                    'pre_period_uplift': None,
                    'campaign_period_uplift': result['uplift'],
                    'post_period_uplift': None,
                    'campaign_incr_orders': incr,
                    'post_incr_orders': None,
                    'run_timestamp': pd.Timestamp.now(tz='Europe/Amsterdam'),
                }
                fsu.write_flash_sales_result(
                    client, row, cfg.AA_RESULTS_TABLE)
                results.append(row)
                print(f'  Uplift: {result["uplift"]:+.4f} '
                      f'({incr:+.0f} orders)')
            else:
                print('  CausalImpact returned no result.')

        except Exception as e:
            print(f'  ERROR: {e}')
            import traceback; traceback.print_exc()
        finally:
            del df_orders
            gc.collect()

elapsed = (time.time() - start_time) / 60
print(f'\n{"="*50}')
print(f'Flash Sales A/A complete in {elapsed:.1f} minutes.')
print(f'{len(results)} successful runs out of {total}.')

## 3. Results Summary

In [ ]:
df_results = client.query(f"""
    SELECT treatment_city, window_start, window_end,
        n_controls, campaign_period_uplift, campaign_incr_orders
    FROM `{cfg.AA_RESULTS_TABLE}`
    WHERE technique = '{cfg.FLASH_SALES_TECHNIQUE}'
    ORDER BY treatment_city, window_start
""").to_dataframe()

print(f'Total runs in BQ: {len(df_results)}')
print(f'\nPer-city summary:')
summary = df_results.groupby('treatment_city').agg(
    n_runs=('campaign_period_uplift', 'count'),
    mean_uplift=('campaign_period_uplift', 'mean'),
    std_uplift=('campaign_period_uplift', 'std'),
    min_uplift=('campaign_period_uplift', 'min'),
    max_uplift=('campaign_period_uplift', 'max'),
    mean_controls=('n_controls', 'mean'),
).round(4)
print(summary.to_string())

# Overall verdict
mean_uplift = df_results['campaign_period_uplift'].mean()
std_uplift = df_results['campaign_period_uplift'].std()
n = len(df_results)

if n > 1:
    from scipy import stats
    ci_95 = stats.t.interval(0.95, df=n-1,
                              loc=mean_uplift,
                              scale=std_uplift / np.sqrt(n))
    ci_contains_zero = ci_95[0] <= 0 <= ci_95[1]

    print(f'\nOverall: mean={mean_uplift:+.4f}, '
          f'95% CI=[{ci_95[0]:+.4f}, {ci_95[1]:+.4f}]')

    if abs(mean_uplift) > cfg.BIAS_THRESHOLD_HARD:
        print('VERDICT: HARD FAIL')
    elif abs(mean_uplift) > cfg.BIAS_THRESHOLD_WARN or not ci_contains_zero:
        print('VERDICT: WARNING')
    else:
        print('VERDICT: PASS')

    # MDE calculation
    print(f'\nMinimum Detectable Effect:')
    uplifts = df_results['campaign_period_uplift'].dropna().values
    mde = fsu.compute_mde(uplifts)
else:
    print('\nInsufficient results for statistical summary.')

## 4. Uplift Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all uplifts
ax = axes[0]
uplifts = df_results['campaign_period_uplift'].dropna()
ax.hist(uplifts, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='Zero (expected)')
ax.axvline(uplifts.mean(), color='blue', linestyle='-', label=f'Mean: {uplifts.mean():+.3f}')
ax.set_xlabel('Campaign Period Uplift')
ax.set_ylabel('Count')
ax.set_title('Flash Sales A/A: Uplift Distribution')
ax.legend()

# Per-city box plot
ax = axes[1]
cities = df_results['treatment_city'].unique()
data = [df_results[df_results['treatment_city']==c]['campaign_period_uplift'].dropna()
        for c in cities]
ax.boxplot(data, labels=cities)
ax.axhline(0, color='red', linestyle='--')
ax.set_ylabel('Campaign Period Uplift')
ax.set_title('Flash Sales A/A: Uplift by City')

plt.tight_layout()
plt.show()